In [1]:
import re
import sys
import pandas as pd

cwe_id = 79

def count_functions(file_content, file_extension):
    if file_extension == 'php':
        pattern = r'function\s+\w+\s*\([^)]*\)\s*(:\s*\S+)?\s*\{'
    elif file_extension in ['ts', 'js']:
        pattern = r'(async\s+)?(?:function\s+\w+\s*\(.*?\)\s*\{|(?:\w+\s*=\s*)?\(.*?\)\s*=>\s*\{|[\w\.]+\s*\([\w\s,]*\)\s*\{)'
    elif file_extension == 'html':
        return 0
    elif file_extension == 'java':
        pattern = r'(public|protected|private|static|\s)+[\w<>\[\],\s]+\s+\w+\s*\([\w\s,<>\[\]]*\)\s*\{'
    elif file_extension == 'go':
        pattern = r'func\s+(?:\([\w\s,*]*\)\s*)?\w+\s*\([^)]*\)\s*\{'
    elif file_extension == 'py':
        pattern = r'(?:@[\w\.]+\s*)*def\s+\w+\s*\([^)]*\)\s*:'
    elif file_extension == 'rb':
        pattern = r'def\s+\w+\s*\([^)]*\)\s*'
    elif file_extension == 'c':
        pattern = r'[\w\s*]+\s+\w+\s*\([\w\s,*]*\)\s*\{'
    else:
        return 0

    matches = re.findall(pattern, file_content)
    return len(matches)

# Read the CSV file
file_path = f'./files_CWE-{cwe_id}.csv'
df = pd.read_csv(file_path)

# Initialize a list to hold the function counts
function_counts = []

# Process each file in the dataframe
for index, row in df.iterrows():
    file_content = row['file_before']
    file_extension = row['file_extension']
    count = count_functions(file_content, file_extension)
    function_counts.append(count)

# Add the function counts to the dataframe
df['function_count'] = function_counts

# Drop files with no functions
print(len(df))
# 


543


In [5]:
df_filtered = df[df['function_count'] == 0] 
# add file before lenghts
df_filtered['file_before_len'] = df_filtered['file_before'].str.len()
df_filtered

C:\Users\adam\AppData\Local\Temp\ipykernel_17228\2168612145.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['file_before_len'] = df_filtered['file_before'].str.len()


,file_id,filename,file_extension,cve_id,file_before,file_after,function_count,file_before_len
0,293,apiserver/version.py,py,CVE-2023-6778,"__version__ = ""1.12.0""\n","__version__ = ""1.13.0""\n",0,23
1,8043,config/version.php,php,CVE-2021-4121,"<?php\n\nreturn [\n\t'appVersion' => '6.3.7',\...","<?php\n\nreturn [\n\t'appVersion' => '6.3.8',\...",0,107
2,8489,config/version.php,php,CVE-2022-3000,"<?php\n\nreturn [\n\t'appVersion' => '6.4.11',...","<?php\n\nreturn [\n\t'appVersion' => '6.4.12',...",0,108
3,3209,app/controllers/pay/payments_controller.rb,rb,CVE-2023-30614,module Pay\n class PaymentsController < Appli...,module Pay\n class PaymentsController < Appli...,0,226
4,7586,src/resources/views/columns/select.blade.php,php,CVE-2018-20962,"{{-- single relationships (1-1, 1-n) --}}\n<sp...","{{-- single relationships (1-1, 1-n) --}}\n<sp...",0,311
...,...,...,...,...,...,...,...,...
499,8338,internal/cmd/web.go,go,CVE-2022-1464,// Copyright 2014 The Gogs Authors. All rights...,// Copyright 2014 The Gogs Authors. All rights...,0,26644
501,73,01article.php,php,CVE-2021-4310,<?PHP\n/* \n\t01-Artikelsystem V3 - Copyright ...,<?PHP\n/* \n\t01-Artikelsystem V3 - Copyright ...,0,30861
514,7630,app/contacts/contact_edit.php,php,CVE-2019-16973,<?php\n/*\n\tFusionPBX\n\tVersion: MPL 1.1\n\n...,<?php\n/*\n\tFusionPBX\n\tVersion: MPL 1.1\n\n...,0,32143
518,8382,htdocs/user/bank.php,php,CVE-2022-2060,<?php\n/* Copyright (C) 2002-2004 Rodolphe Qu...,<?php\n/* Copyright (C) 2002-2004 Rodolphe Qu...,0,34304


In [ ]:
df = df[df['function_count'] > 0]
print(len(df))


# Calculate the average function count across all files
average_function_count = df['function_count'].mean()

# Calculate the average function size by dividing the file sizes by the number of functions
# First, we need to calculate the size of each file (in terms of characters)
df['file_size'] = df['file_before'].apply(len)

# Avoid division by zero by setting function count to 1 where it is zero
df['adjusted_function_count'] = df['function_count'].apply(lambda x: x if x != 0 else 1)

# Calculate average function size
df['average_function_size'] = df['file_size'] / df['adjusted_function_count']

# Calculate the average function size across all files
average_function_size = df['average_function_size'].mean()

print('average_function_count:', average_function_count)
print('average_function_size:', average_function_size)

# Calculate the median function count across the filtered files
median_function_count = df['function_count'].median()

# Calculate the median function size across the filtered files
median_function_size = df['average_function_size'].median()

print('median_function_count:', median_function_count)
print('median_function_size:', median_function_size)